 # RBM Trained on Ising Snapshots sampled at criticality

In [1]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import dataloader as dl
import mcfile as mcf
import matplotlib.pyplot as plt

In [2]:
# ─── Core functions ───────────────────────────────────────────────────────────

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def init_rbm(n_visible, n_hidden, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    scale = np.sqrt(2 / (n_visible + n_hidden))
    W  = rng.standard_normal((n_hidden, n_visible)) * scale
    bv = np.zeros(n_visible)
    bh = np.zeros(n_hidden)
    return W, bv, bh

def hidden_probs(W, bh, v):
    return sigmoid(v @ W.T + bh)          # (batch, n_hidden)

def visible_probs(W, bv, h):
    return sigmoid(h @ W + bv)            # (batch, n_visible)

def sample(probs, rng):
    return (rng.random(probs.shape) < probs).astype(np.float32)

# ─── Contrastive Divergence ───────────────────────────────────────────────────

def cd_step(W, bv, bh, v0, k=1, lr=0.01, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    # Positive phase
    ph0 = hidden_probs(W, bh, v0)
    h0  = sample(ph0, rng)

    # Negative phase (k Gibbs steps)
    vk, hk, phk = v0.copy(), h0.copy(), ph0.copy()
    for _ in range(k):
        pvk = visible_probs(W, bv, hk)
        vk  = sample(pvk, rng)
        phk = hidden_probs(W, bh, vk)
        hk  = sample(phk, rng)

    # Gradients (averaged over batch)
    dW  = (ph0.T @ v0 - phk.T @ vk) / len(v0)
    dbv = (v0  - vk).mean(axis=0)
    dbh = (ph0 - phk).mean(axis=0)

    W  = W  + lr * dW
    bv = bv + lr * dbv
    bh = bh + lr * dbh

    # Reconstruction MSE
    pv_recon = visible_probs(W, bv, hk)
    loss = np.mean((v0 - pv_recon) ** 2)

    return W, bv, bh, loss



In [ ]:
# ─── Training loop ────────────────────────────────────────────────────────────

def train(W, bv, bh, data, epochs=200, batch_size=32, k=1, lr=0.01, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    n = len(data)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        data = data[idx]
        losses = []
        for start in range(0, n, batch_size):
            batch = data[start : start + batch_size].astype(np.float32)
            W, bv, bh, loss = cd_step(W, bv, bh, batch, k=k, lr=lr, rng=rng)
            losses.append(loss)
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:4d}/{epochs}  MSE={np.mean(losses):.4f}")
    return W, bv, bh

# ─── Generation (Gibbs sampling) ──────────────────────────────────────────────

def generate(W, bv, bh, n_samples=8, gibbs_steps=100, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    v = rng.integers(0, 2, size=(n_samples, W.shape[1])).astype(np.float32)
    for _ in range(gibbs_steps):
        ph = hidden_probs(W, bh, v)
        h  = sample(ph, rng)
        pv = visible_probs(W, bv, h)
        v  = sample(pv, rng)
    return v

In [7]:
b_c = 0.440687
sector = 'AP'
sizes = [24,32,48,64]
L = sizes[1]

datadir = "ising_critical_scan_long"
data = mcf.build_dataframe(datadir)
bc_num = data['beta'][np.argmin(np.abs(np.array(data['beta'])-b_c))]
snapshots = data.query('beta == @bc_num and sector == @sector and Lx == @L and Ly == @L')['spins']
snapshots = (np.array(snapshots))[0]
print(bc_num, snapshots.shape)

Processing files: 100%|█| 320/320 [00:22<00:00, 13.96file/s, L64_beta0.449000_se

0.4408947368421053 (20000, 32, 32)


In [11]:
# ─── Main ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rng  = np.random.default_rng(42)

    data = snapshots 
    data = data.reshape(len(data), -1)       # flatten spatial dims if present
    data = (data > 0).astype(np.float32)  # binarize

    n_visible = data.shape[1]
    n_hidden  = 64                           

    W, bv, bh = init_rbm(n_visible, n_hidden, rng=rng)
    W, bv, bh = train(W, bv, bh, data, epochs=200, batch_size=32, k=1, lr=0.01, rng=rng)

    samples = generate(W, bv, bh, n_samples=8, gibbs_steps=100, rng=rng)
    print("Generated shape:", samples.shape)  # (8, n_visible)

    np.save("generated.npy", samples)

Epoch   10/200  MSE=0.1656
Epoch   20/200  MSE=0.1575
Epoch   30/200  MSE=0.1539
Epoch   40/200  MSE=0.1516
Epoch   50/200  MSE=0.1495
Epoch   60/200  MSE=0.1476
Epoch   70/200  MSE=0.1458
Epoch   80/200  MSE=0.1443
Epoch   90/200  MSE=0.1430
Epoch  100/200  MSE=0.1417
Epoch  110/200  MSE=0.1405
Epoch  120/200  MSE=0.1390
Epoch  130/200  MSE=0.1378
Epoch  140/200  MSE=0.1366
Epoch  150/200  MSE=0.1356
Epoch  160/200  MSE=0.1346
Epoch  170/200  MSE=0.1338
Epoch  180/200  MSE=0.1330
Epoch  190/200  MSE=0.1322
Epoch  200/200  MSE=0.1316
Generated shape: (8, 1024)


In [14]:
import numpy as np
from itertools import product
from dataclasses import dataclass, field
from typing import Any

# ─── Result container (plain dataclass, no methods) ───────────────────────────

@dataclass
class RunResult:
    params: dict
    final_loss: float
    loss_curve: list[float]
    W: Any
    bv: Any
    bh: Any

# ─── Single run ───────────────────────────────────────────────────────────────

def run_once(data, n_hidden, epochs, batch_size, k, lr, rng):
    W, bv, bh = init_rbm(data.shape[1], n_hidden, rng=rng)
    losses = []
    n = len(data)
    for _ in range(epochs):
        idx = rng.permutation(n)
        batch_losses = []
        for start in range(0, n, batch_size):
            batch = data[idx[start : start + batch_size]].astype(np.float32)
            W, bv, bh, loss = cd_step(W, bv, bh, batch, k=k, lr=lr, rng=rng)
            batch_losses.append(loss)
        losses.append(float(np.mean(batch_losses)))
    return W, bv, bh, losses

# ─── Grid scan ────────────────────────────────────────────────────────────────

def scan(data, grid, epochs=100, batch_size=32, seed=42):
    """
    data  : np.ndarray, shape (n_samples, n_visible), binary float32
    grid  : dict of lists, e.g.
            {"n_hidden": [32, 64], "lr": [0.01, 0.05], "k": [1, 3]}
    epochs, batch_size applied to every run.
    Returns list[RunResult] sorted by final_loss ascending.
    """
    keys   = list(grid.keys())
    combos = list(product(*[grid[k] for k in keys]))
    total  = len(combos)
    results = []

    print(f"Scanning {total} combinations × {epochs} epochs each\n")

    for i, values in enumerate(combos, 1):
        params = dict(zip(keys, values))
        rng    = np.random.default_rng(seed)

        print(f"[{i}/{total}] {params} ...", end=" ", flush=True)
        W, bv, bh, curve = run_once(
            data,
            n_hidden   = params.get("n_hidden",   32),
            epochs     = epochs,
            batch_size = params.get("batch_size", batch_size),
            k          = params.get("k",          1),
            lr         = params.get("lr",         0.01),
            rng        = rng,
        )
        final = curve[-1]
        print(f"loss={final:.4f}")
        results.append(RunResult(params=params, final_loss=final,
                                 loss_curve=curve, W=W, bv=bv, bh=bh))

    return sorted(results, key=lambda r: r.final_loss)

# ─── Summary ──────────────────────────────────────────────────────────────────

def summarize(results, top_n=5):
    sep = "─" * 62
    print(f"\n{'HYPERPARAMETER SCAN RESULTS':^62}")
    print(sep)

    # top runs
    print(f"\n Top {top_n} runs (by final reconstruction MSE):\n")
    header_params = list(results[0].params.keys())
    col_w = max(len(k) for k in header_params) + 2
    header = "  rank  loss    " + "  ".join(f"{k:<{col_w}}" for k in header_params)
    print(header)
    print("  " + "─" * (len(header) - 2))
    for rank, r in enumerate(results[:top_n], 1):
        vals = "  ".join(f"{r.params[k]:<{col_w}}" for k in header_params)
        print(f"  {rank:<5} {r.final_loss:.4f}  {vals}")

    # per-param sensitivity
    print(f"\n Per-parameter sensitivity (mean final loss):\n")
    for key in results[0].params:
        unique_vals = sorted({r.params[key] for r in results})
        print(f"  {key}:")
        for v in unique_vals:
            subset = [r.final_loss for r in results if r.params[key] == v]
            print(f"    {str(v):<10}  mean={np.mean(subset):.4f}  "
                  f"min={np.min(subset):.4f}  max={np.max(subset):.4f}")
        print()

    # convergence snapshot (loss at 25%, 50%, 75%, 100% of epochs)
    print(f" Convergence profile — best run:\n")
    best = results[0]
    n    = len(best.loss_curve)
    checkpoints = [int(n * f) - 1 for f in (0.25, 0.5, 0.75, 1.0)]
    for cp in checkpoints:
        pct = int((cp + 1) / n * 100)
        print(f"   epoch {cp+1:>4} ({pct:>3}%)  loss={best.loss_curve[cp]:.4f}")

    print(f"\n{sep}")
    print(f" Best config: {best.params}  →  loss={best.final_loss:.4f}")
    print(sep)

    return results[0]   # return best RunResult for convenience

# ─── Usage ────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    data = snapshots 
    data = data.reshape(len(data), -1)       # flatten spatial dims if present
    data = (data > 0).astype(np.float32)  # binarize

    grid = {
        "n_hidden": [32, 64, 128],
        "lr":       [0.005, 0.01, 0.05],
        "k":        [1, 3],
    }

    results = scan(data, grid, epochs=100, batch_size=32, seed=42)
    best    = summarize(results, top_n=5)

    # best weights are ready to use directly
    samples = generate(best.W, best.bv, best.bh, n_samples=8)
    np.save("generated.npy", samples)

Scanning 18 combinations × 100 epochs each

[1/18] {'n_hidden': 32, 'lr': 0.005, 'k': 1} ... loss=0.1690
[2/18] {'n_hidden': 32, 'lr': 0.005, 'k': 3} ... loss=0.1656
[3/18] {'n_hidden': 32, 'lr': 0.01, 'k': 1} ... loss=0.1668
[4/18] {'n_hidden': 32, 'lr': 0.01, 'k': 3} ... loss=0.1609
[5/18] {'n_hidden': 32, 'lr': 0.05, 'k': 1} ... 

/tmp/ipykernel_13626/1929027351.py:4: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-x))


loss=0.1560
[6/18] {'n_hidden': 32, 'lr': 0.05, 'k': 3} ... loss=0.1371
[7/18] {'n_hidden': 64, 'lr': 0.005, 'k': 1} ... loss=0.1493
[8/18] {'n_hidden': 64, 'lr': 0.005, 'k': 3} ... loss=0.1488
[9/18] {'n_hidden': 64, 'lr': 0.01, 'k': 1} ... loss=0.1415
[10/18] {'n_hidden': 64, 'lr': 0.01, 'k': 3} ... loss=0.1382
[11/18] {'n_hidden': 64, 'lr': 0.05, 'k': 1} ... loss=0.1231
[12/18] {'n_hidden': 64, 'lr': 0.05, 'k': 3} ... loss=0.1239
[13/18] {'n_hidden': 128, 'lr': 0.005, 'k': 1} ... loss=0.1290
[14/18] {'n_hidden': 128, 'lr': 0.005, 'k': 3} ... loss=0.1343
[15/18] {'n_hidden': 128, 'lr': 0.01, 'k': 1} ... loss=0.1207
[16/18] {'n_hidden': 128, 'lr': 0.01, 'k': 3} ... loss=0.1256
[17/18] {'n_hidden': 128, 'lr': 0.05, 'k': 1} ... loss=0.1102
[18/18] {'n_hidden': 128, 'lr': 0.05, 'k': 3} ... loss=0.1161

                 HYPERPARAMETER SCAN RESULTS                  
──────────────────────────────────────────────────────────────

 Top 5 runs (by final reconstruction MSE):

  rank  loss    n

In [15]:
import numpy as np

# ─── Free energy F(v; θ) ─────────────────────────────────────────────────────
# F(v) = -v @ bv - sum_j log(1 + exp(bh_j + W_j @ v))
# shape: (batch,) → mean → scalar

def free_energy(W, bv, bh, v):
    """F(v; θ) per sample, shape (batch,)"""
    visible_term = v @ bv                                    # (batch,)
    hidden_input = v @ W.T + bh                             # (batch, n_hidden)
    # log(1 + exp(x)) done stably
    hidden_term  = np.logaddexp(0, hidden_input).sum(axis=1) # (batch,)
    return -visible_term - hidden_term


# ─── Annealed Importance Sampling for log Z ──────────────────────────────────
# Bridges p_0 (uniform/base) to p_N (model) via β_t schedule.
# Returns a single scalar estimate of log Z.

def log_z_ais(W, bv, bh, n_chains=100, n_betas=500, rng=None):
    """
    Estimate log Z via AIS.
    Intermediate distributions: p_t(v) ∝ exp(β_t · F̃(v))
    where F̃ includes only the visible bias + hidden terms (RBM factorises nicely).

    Base distribution p_0: uniform Bernoulli(0.5) over visible units.
      log Z_0 = n_visible * log 2.
    """
    if rng is None:
        rng = np.random.default_rng()

    n_visible = W.shape[1]
    betas     = np.linspace(0, 1, n_betas)

    # log Z of base distribution
    log_z0 = n_visible * np.log(2)

    # Initialise chains from base (uniform Bernoulli)
    v = rng.integers(0, 2, size=(n_chains, n_visible)).astype(np.float32)

    # Accumulate log importance weights
    log_w = np.zeros(n_chains)

    for i in range(1, n_betas):
        b_prev, b_curr = betas[i - 1], betas[i]

        # log contribution: (β_curr - β_prev) · (-F(v))  [since p_t ∝ exp(-β_t F)]
        # F is already negative inside (we want the exponent of the unnorm. prob.)
        fe = free_energy(W, bv, bh, v)          # (n_chains,)  lower = more probable
        log_w += (b_curr - b_prev) * (-fe)

        # Gibbs transition at β_curr to keep chains valid
        # hidden
        ph = sigmoid(b_curr * (v @ W.T + bh))
        h  = (rng.random((n_chains, W.shape[0])) < ph).astype(np.float32)
        # visible
        pv = sigmoid(b_curr * (h @ W + bv))
        v  = (rng.random((n_chains, n_visible)) < pv).astype(np.float32)

    # log Z = log Z_0 + log mean exp(log_w)  (log-sum-exp for stability)
    log_w_max = log_w.max()
    log_z_est = log_z0 + log_w_max + np.log(np.exp(log_w - log_w_max).mean())
    return float(log_z_est)


# ─── All four metrics ────────────────────────────────────────────────────────

def compute_metrics(W, bv, bh, data, k, batch_size=32,
                    ais_chains=200, ais_betas=500, rng=None):
    """
    Returns dict with:
      log_likelihood  – metric 1: avg log p(v) = avg F(v) - log Z
      mean_free_energy – metric 2: avg F(v) across batches
      cd_loss          – metric 3: avg [ F(v0) - F(vk) ] across batches
      recon_error      – metric 4: avg |v0 - v1|^2 across batches
    All batch-averaged exactly as in Eqs. 20-23.
    """
    if rng is None:
        rng = np.random.default_rng()

    n = len(data)
    log_z = log_z_ais(W, bv, bh,
                      n_chains=ais_chains, n_betas=ais_betas, rng=rng)

    batch_fe    = []   # metric 2
    batch_loss  = []   # metric 3
    batch_recon = []   # metric 4

    for start in range(0, n, batch_size):
        v0 = data[start : start + batch_size].astype(np.float32)

        # ── metric 2: mean F(v0) ──
        fe0 = free_energy(W, bv, bh, v0)
        batch_fe.append(fe0.mean())

        # ── metrics 3 & 4: need v1 and vk ──
        # one Gibbs step → v1 (for recon error)
        ph1 = sigmoid(v0 @ W.T + bh)
        h1  = (rng.random(ph1.shape) < ph1).astype(np.float32)
        pv1 = sigmoid(h1 @ W + bv)
        v1  = (rng.random(pv1.shape) < pv1).astype(np.float32)

        recon = np.mean((v0 - v1) ** 2)
        batch_recon.append(recon)

        # k Gibbs steps → vk (for CD loss)
        vk = v1.copy()
        for _ in range(k - 1):
            phk = sigmoid(vk @ W.T + bh)
            hk  = (rng.random(phk.shape) < phk).astype(np.float32)
            pvk = sigmoid(hk @ W + bv)
            vk  = (rng.random(pvk.shape) < pvk).astype(np.float32)

        fek = free_energy(W, bv, bh, vk)
        batch_loss.append((fe0 - fek).mean())

    mean_fe    = float(np.mean(batch_fe))
    cd_loss    = float(np.mean(batch_loss))
    recon_err  = float(np.mean(batch_recon))
    log_like   = mean_fe - log_z          # Eq. 21

    return {
        "log_likelihood":   log_like,
        "mean_free_energy": mean_fe,
        "cd_loss":          cd_loss,
        "recon_error":      recon_err,
        "log_z":            log_z,
    }


# ─── Training loop with monitoring ───────────────────────────────────────────

def train_monitored(W, bv, bh, data, epochs=200, batch_size=32, k=1, lr=0.01,
                    monitor_every=10, ais_chains=200, ais_betas=500, rng=None):
    """
    Trains the RBM and computes all four metrics every `monitor_every` epochs.
    Returns (W, bv, bh, history) where history is a list of dicts.
    """
    if rng is None:
        rng = np.random.default_rng()

    n       = len(data)
    history = []

    _pad = len(str(epochs))
    print(f"{'epoch':>{_pad}}  {'log_like':>10}  {'mean_F':>10}  {'cd_loss':>10}  {'recon_err':>10}  {'log_Z':>10}")
    print("─" * (_pad + 57))

    for epoch in range(1, epochs + 1):
        idx = rng.permutation(n)
        for start in range(0, n, batch_size):
            batch = data[idx[start : start + batch_size]].astype(np.float32)
            W, bv, bh, _ = cd_step(W, bv, bh, batch, k=k, lr=lr, rng=rng)

        if epoch % monitor_every == 0 or epoch == 1:
            m = compute_metrics(W, bv, bh, data, k=k, batch_size=batch_size,
                                ais_chains=ais_chains, ais_betas=ais_betas, rng=rng)
            m["epoch"] = epoch
            history.append(m)
            print(f"{epoch:>{_pad}}  "
                  f"{m['log_likelihood']:>10.4f}  "
                  f"{m['mean_free_energy']:>10.4f}  "
                  f"{m['cd_loss']:>10.4f}  "
                  f"{m['recon_error']:>10.4f}  "
                  f"{m['log_z']:>10.4f}")

    return W, bv, bh, history

In [38]:
remove_before = 7000

# ─── Usage ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rng  = np.random.default_rng(42)
    data = snapshots[remove_before:]
    data = data.reshape(len(data), -1)
    data = (data > 0.0).astype(np.float32)

    W, bv, bh = init_rbm(data.shape[1], n_hidden=256, rng=rng)
    W, bv, bh, history = train_monitored(
        W, bv, bh, data,
        epochs        = 200,
        batch_size    = 32,
        k             = 3,
        lr            = 0.05,
        monitor_every = 10,
        ais_chains    = 50,   # ↑ for more accurate log Z, ↓ for speed
        ais_betas     = 3000,
    )

epoch    log_like      mean_F     cd_loss   recon_err       log_Z
────────────────────────────────────────────────────────────
  1  -1678.9917   -565.3874     16.7498      0.3048   1113.6043
 10  -2618.1061  -1081.3694     -4.7082      0.2295   1536.7367
 20  -3191.0823  -1380.5813      7.5045      0.2088   1810.5010
 30  -3440.1295  -1504.0685     14.3534      0.2013   1936.0610
 40  -3678.1585  -1623.9910     18.5937      0.1941   2054.1675
 50  -3773.0123  -1672.1437     21.4160      0.1911   2100.8685
 60  -3878.6844  -1720.9303     26.3232      0.1886   2157.7541
 70  -3931.6394  -1748.8751     26.5021      0.1878   2182.7643
 80  -3991.3484  -1782.6842     21.6883      0.1852   2208.6642
 90  -4014.0511  -1797.9080     21.7944      0.1846   2216.1431
100  -4092.8452  -1813.5256     35.9404      0.1847   2279.3196
110  -4087.3284  -1831.2455     30.5996      0.1831   2256.0829
120  -4051.0145  -1820.3924     25.3121      0.1828   2230.6221
130  -4120.9537  -1846.6965     31.2117  

In [30]:
import pickle

# Save (Serialize) data to a file
with open('pprbmnh256.pkl', 'wb') as file:
    pickle.dump({'W': W, 'bv': bv, 'bh': bh, 'history': history}, file)


In [39]:
def log_like_sanity(W, bv, bh, data, rng=None):
    # Exact lower bound via AIS, plus naive baselines for comparison
    n_visible = W.shape[1]

    random_baseline = n_visible * np.log(0.5)

    # Empirical visible bias model: p(v_i) = mean(data_i), independent
    p_data = data.mean(axis=0).clip(1e-6, 1 - 1e-6)
    empirical_baseline = (data * np.log(p_data) + (1 - data) * np.log(1 - p_data)).sum(axis=1).mean()

    print(f"  n_visible         : {n_visible}")
    print(f"  random baseline   : {random_baseline:.2f}   (Bernoulli 0.5, no structure)")
    print(f"  empirical baseline: {empirical_baseline:.2f}   (per-pixel marginals, no correlations)")
    print(f"  expected RBM range: [{2*random_baseline:.2f}, 0]")

In [40]:
log_like_sanity(W, bv, bh, data[:256])

  n_visible         : 1024
  random baseline   : -709.78   (Bernoulli 0.5, no structure)
  empirical baseline: -708.07   (per-pixel marginals, no correlations)
  expected RBM range: [-1419.57, 0]


In [41]:
?np.logaddexp

Signature:       np.logaddexp(*args, **kwargs)
Type:            ufunc
String form:     <ufunc 'logaddexp'>
File:            ~/miniconda/envs/jax-cpu/lib/python3.11/site-packages/numpy/__init__.py
Docstring:      
logaddexp(x1, x2, /, out=None, *, where=True, casting='same_kind', order='K', dtype=None, subok=True[, signature])

Logarithm of the sum of exponentiations of the inputs.

Calculates ``log(exp(x1) + exp(x2))``. This function is useful in
statistics where the calculated probabilities of events may be so small
as to exceed the range of normal floating point numbers.  In such cases
the logarithm of the calculated probability is stored. This function
allows adding probabilities stored in such a fashion.

Parameters
----------
x1, x2 : array_like
    Input values.
    If ``x1.shape != x2.shape``, they must be broadcastable to a common
    shape (which becomes the shape of the output).
out : ndarray, None, or tuple of ndarray and None, optional
    A location into which the result i